# Proyecto Final Data Science
## Condiciones oceanográficas y distribución de fauna marina

### Planteamiento inicial

El objetivo de este proyecto es analizar cómo las variaciones en las condiciones del océano están relacionadas con los cambios en la distribución geográfica de la fauna marina.

Trabajaremos principalmente con datos históricos del programa CalCOFI, que contienen información oceanográfica como temperatura, salinidad, oxígeno, profundidad, nutrientes, localización y fecha de las mediciones.

Como primera especie de estudio proponemos la sardina del Pacífico (cabría posibilidad de estudiar alguna mas pero creo que debemos centrarnos en una especie). Analizaremos si los cambios en las condiciones ambientales están asociados con cambios en las zonas donde aparece la especie.

#### Objetivo de Machine Learning

Construir un modelo capaz de estimar la probabilidad de presencia de la especie a partir de las condiciones oceanográficas.





### SQL y unión de datos

En esta fase vamos a cargar `bottle.csv` y `cast.csv` en una base de datos.

El objetivo es aprender a consultar ambas tablas y unirlas mediante `Cst_Cnt` para obtener en un mismo dataset las condiciones del océano, la fecha y la localización de cada muestra.


In [2]:
import pandas as pd
import sqlite3


In [3]:
# Cargamos los datasets

bottle = pd.read_csv("../../data/raw/bottle.csv")
cast = pd.read_csv("../../data/raw/cast.csv")

/tmp/ipykernel_1413/2947451975.py:3: DtypeWarning: Columns (0: IncTim, 1: DIC Quality Comment) have mixed types. Specify dtype option on import or set low_memory=False.
  bottle = pd.read_csv("../../data/raw/bottle.csv")
/tmp/ipykernel_1413/2947451975.py:4: DtypeWarning: Columns (0: Data_Or, 1: Cruz_Num, 2: Inc_Str, 3: Inc_End, 4: PST_LAN, 5: Civil_T) have mixed types. Specify dtype option on import or set low_memory=False.
  cast = pd.read_csv("../../data/raw/cast.csv")


In [4]:
# Comprobamos el tamaño de los datasets

print("Bottle:", bottle.shape)
print("Cast:", cast.shape)


Bottle: (864863, 74)
Cast: (34404, 61)


In [5]:
# Creamos la base de datos SQL

conn = sqlite3.connect("../../data/calcofi.db")

In [6]:
# Guardamos los datasets como tablas SQL

bottle.to_sql("bottle", conn, if_exists="replace", index=False)
cast.to_sql("cast", conn, if_exists="replace", index=False)

34404

In [7]:
# Vemos las primeras filas de la tabla bottle

pd.read_sql_query("""
SELECT *
FROM bottle
LIMIT 5;
""", conn)

,Cst_Cnt,Btl_Cnt,Sta_ID,Depth_ID,Depthm,T_degC,Salnty,O2ml_L,STheta,O2Sat,...,R_PHAEO,R_PRES,R_SAMP,DIC1,DIC2,TA1,TA2,pH2,pH1,DIC Quality Comment
0,1,1,054.0 056.0,19-4903CR-HY-060-0930-05400560-0000A-3,0,10.50,33.440,None,25.649,None,...,None,0,None,None,None,None,None,None,None,None
1,1,2,054.0 056.0,19-4903CR-HY-060-0930-05400560-0008A-3,8,10.46,33.440,None,25.656,None,...,None,8,None,None,None,None,None,None,None,None
2,1,3,054.0 056.0,19-4903CR-HY-060-0930-05400560-0010A-7,10,10.46,33.437,None,25.654,None,...,None,10,None,None,None,None,None,None,None,None
3,1,4,054.0 056.0,19-4903CR-HY-060-0930-05400560-0019A-3,19,10.45,33.420,None,25.643,None,...,None,19,None,None,None,None,None,None,None,None
4,1,5,054.0 056.0,19-4903CR-HY-060-0930-05400560-0020A-7,20,10.45,33.421,None,25.643,None,...,None,20,None,None,None,None,None,None,None,None


In [8]:
# Vemos las primeras filas de la tabla cast

pd.read_sql_query("""
SELECT *
FROM cast
LIMIT 5;
""", conn)

,Cst_Cnt,Cruise_ID,Cruise,Cruz_Sta,DbSta_ID,Cast_ID,Sta_ID,Quarter,Sta_Code,Distance,...,Wave_Prd,Wind_Dir,Wind_Spd,Barometer,Dry_T,Wet_T,Wea,Cloud_Typ,Cloud_Amt,Visibility
0,1,1949-03-01-C-31CR,194903,19490305400560,5400560,19-4903CR-HY-060-0930-05400560,054.0 056.0,1,NST,None,...,None,23.0,18.0,None,None,None,2.0,None,None,None
1,2,1949-03-01-C-31CR,194903,19490305200750,5200750,19-4903CR-HY-060-2112-05200750,052.0 075.0,1,NST,None,...,None,16.0,5.0,None,None,None,4.0,None,None,None
2,3,1949-03-01-C-31CR,194903,19490305100850,5100850,19-4903CR-HY-061-0354-05100850,051.0 085.0,1,NST,None,...,None,23.0,5.0,None,None,None,6.0,None,None,None
3,4,1949-03-01-C-31CR,194903,19490305000950,5000950,19-4903CR-HY-061-1042-05000950,050.0 095.0,1,NST,None,...,None,18.0,8.0,None,None,None,2.0,None,None,None
4,5,1949-03-01-C-31CR,194903,19490305001040,5001040,19-4903CR-HY-061-1706-05001040,050.0 104.0,1,NST,None,...,None,27.0,13.0,None,None,None,7.0,None,None,None


In [9]:
# Comprobamos cuántas mediciones tiene cada muestreo

pd.read_sql_query("""
SELECT Cst_Cnt, COUNT(*) AS mediciones
FROM bottle
GROUP BY Cst_Cnt
LIMIT 10;
""", conn)

,Cst_Cnt,mediciones
0,1,29
1,2,32
2,3,31
3,4,31
4,5,26
5,6,34
6,7,33
7,8,31
8,9,32
9,10,32


### Relación entre Cast y Bottle

La tabla `cast` (**muestreo / lance oceanográfico**) representa cada muestreo realizado en el océano.

La tabla `bottle` (**botella / muestra de agua**) contiene las distintas mediciones tomadas dentro de cada muestreo, normalmente a diferentes profundidades.

Por eso `bottle` tiene muchas más filas que `cast`.

Ejemplo:

- 1 registro en `cast`
- puede tener 20, 30 o más mediciones en `bottle`

La relación entre ambas tablas es:

**1 Cast (muestreo) → muchas Bottle (muestras de agua)**

In [10]:
# Creamos una tabla unida con Bottle y Cast

pd.read_sql_query("""
SELECT *
FROM bottle AS b
INNER JOIN cast AS c
ON b.Cst_Cnt = c.Cst_Cnt
LIMIT 5;
""", conn)

,Cst_Cnt,Btl_Cnt,Sta_ID,Depth_ID,Depthm,T_degC,Salnty,O2ml_L,STheta,O2Sat,...,Wave_Prd,Wind_Dir,Wind_Spd,Barometer,Dry_T,Wet_T,Wea,Cloud_Typ,Cloud_Amt,Visibility
0,1,1,054.0 056.0,19-4903CR-HY-060-0930-05400560-0000A-3,0,10.50,33.440,None,25.649,None,...,None,23.0,18.0,None,None,None,2.0,None,None,None
1,1,2,054.0 056.0,19-4903CR-HY-060-0930-05400560-0008A-3,8,10.46,33.440,None,25.656,None,...,None,23.0,18.0,None,None,None,2.0,None,None,None
2,1,3,054.0 056.0,19-4903CR-HY-060-0930-05400560-0010A-7,10,10.46,33.437,None,25.654,None,...,None,23.0,18.0,None,None,None,2.0,None,None,None
3,1,4,054.0 056.0,19-4903CR-HY-060-0930-05400560-0019A-3,19,10.45,33.420,None,25.643,None,...,None,23.0,18.0,None,None,None,2.0,None,None,None
4,1,5,054.0 056.0,19-4903CR-HY-060-0930-05400560-0020A-7,20,10.45,33.421,None,25.643,None,...,None,23.0,18.0,None,None,None,2.0,None,None,None


## Estado actual del proyecto

Hasta este punto hemos:

- Cargado los datasets `bottle.csv` y `cast.csv`.
- Creado una base de datos SQLite.
- Consultado ambas tablas mediante SQL.
- Comprobado la relación entre `cast` y `bottle` usando `Cst_Cnt`.
- Verificado que la unión mantiene correctamente las 864.863 muestras.

### Siguiente paso: selección de variables

Antes de continuar con la estadística descriptiva y el EDA, debemos decidir qué variables son realmente útiles para nuestro objetivo:

**Analizar cómo las variaciones en las condiciones del océano están relacionadas con los cambios en la distribución geográfica de la fauna marina.**

La selección de variables se debatirá en equipo teniendo en cuenta:

- Relación con el objetivo.
- Cantidad y calidad de los datos.
- Valores nulos.
- Información repetida o redundante.
- Utilidad para el análisis posterior.

**Pendiente: decidir en equipo las variables que formarán nuestro dataset de trabajo.**